# Notebook 2 — Financial Health Scoring
Develop and validate the health scoring model interactively. Loads clean CSVs, computes all dimension scores, runs sensitivity analysis, and exports final scores.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120})

DATA = Path('../data/clean')
pl = pd.read_csv(DATA / 'profitandloss.csv')
bs = pd.read_csv(DATA / 'balancesheet.csv')
cf = pd.read_csv(DATA / 'cashflow.csv')
co = pd.read_csv(DATA / 'companies.csv')
an = pd.read_csv(DATA / 'analysis.csv')

# Non-TTM annual rows only
pl = pl[pl['is_ttm'] == False].copy()
bs = bs[bs['is_ttm'] == False].copy()
cf = cf[cf['is_ttm'] == False].copy()

print('Data loaded successfully.')
print(f'P&L rows: {len(pl)} | BS rows: {len(bs)} | CF rows: {len(cf)}')

## Step 1 — Scoring helpers

In [ ]:
def minmax(series, low, high):
    clipped = series.clip(lower=low, upper=high)
    return ((clipped - low) / (high - low)) * 100

def percentile_score(series):
    return series.rank(pct=True) * 100

HEALTH_BANDS = [(85, 'EXCELLENT'), (70, 'GOOD'), (50, 'AVERAGE'), (35, 'WEAK'), (0, 'POOR')]

def assign_label(score):
    for threshold, label in HEALTH_BANDS:
        if score >= threshold:
            return label
    return 'POOR'

print('Helpers defined.')

## Step 2 — Compute all dimension scores

In [ ]:
# --- Profitability ---
pl_latest = pl.sort_values('fiscal_year').groupby('company_id').last().reset_index()
s_profit = pd.DataFrame({'symbol': pl_latest['company_id']})
s_profit['npm']  = minmax(pl_latest['net_profit_margin_pct'].fillna(0), -20, 40)
s_profit['opm']  = minmax((pl_latest['operating_profit'] / pl_latest['sales'].replace(0, np.nan) * 100).fillna(0), -10, 50)
s_profit['ic']   = minmax(pl_latest['interest_coverage'].fillna(0).clip(upper=30), 0, 30)
s_profit['profitability_score'] = (s_profit['npm']*0.4 + s_profit['opm']*0.4 + s_profit['ic']*0.2).round(2)

# --- Growth ---
an3 = an[an['period'] == '3Y'][['company_id','compounded_sales_growth_pct','compounded_profit_growth_pct']].copy()
pl_sorted = pl.sort_values('fiscal_year')
pl_sorted['yoy'] = pl_sorted.groupby('company_id')['sales'].pct_change() * 100
yoy = pl_sorted.groupby('company_id')['yoy'].mean().reset_index()
yoy.columns = ['symbol', 'avg_yoy']
df_g = yoy.merge(an3, left_on='symbol', right_on='company_id', how='left')
df_g['csg'] = minmax(df_g['compounded_sales_growth_pct'].fillna(df_g['avg_yoy'].fillna(0)), -5, 30)
df_g['cpg'] = minmax(df_g['compounded_profit_growth_pct'].fillna(0), -10, 40)
df_g['growth_score'] = (df_g['csg']*0.5 + df_g['cpg']*0.5).round(2)
s_growth = df_g[['symbol','growth_score']]

# --- Leverage ---
bs_latest = bs.sort_values('fiscal_year').groupby('company_id').last().reset_index()
dte = bs_latest['debt_to_equity'].fillna(5).clip(0, 10)
eqr = bs_latest['equity_ratio'].fillna(0).clip(0, 1)
s_lev = pd.DataFrame({'symbol': bs_latest['company_id']})
s_lev['dte_score']    = minmax(-dte, -10, 0)
s_lev['equity_score'] = minmax(eqr, 0, 1)
s_lev['leverage_score'] = (s_lev['dte_score']*0.6 + s_lev['equity_score']*0.4).round(2)

# --- Cashflow ---
cf_latest = cf.sort_values('fiscal_year').groupby('company_id').last().reset_index()
df_cf = cf_latest.merge(pl_latest[['company_id','net_profit']], on='company_id', how='left')
s_cf = pd.DataFrame({'symbol': df_cf['company_id']})
s_cf['ocf'] = minmax(df_cf['operating_activity'].fillna(0), -5000, 50000)
s_cf['fcf'] = minmax(df_cf['free_cash_flow'].fillna(0), -10000, 40000)
ccr = (df_cf['operating_activity'] / df_cf['net_profit'].replace(0, np.nan)).fillna(0).clip(-2, 5)
s_cf['ccr'] = minmax(ccr, -2, 5)
s_cf['cashflow_score'] = (s_cf['ocf']*0.4 + s_cf['fcf']*0.3 + s_cf['ccr']*0.3).round(2)

# --- Dividend ---
div_pos = pl[pl['dividend_payout'] > 0]
div_cons = div_pos.groupby('company_id').size().reset_index(name='div_years')
df_div = pl_latest[['company_id','dividend_payout']].merge(div_cons, on='company_id', how='left')
df_div['div_years'] = df_div['div_years'].fillna(0)
s_div = pd.DataFrame({'symbol': df_div['company_id']})
s_div['consistency'] = minmax(df_div['div_years'], 0, 12)
s_div['payout']      = minmax(df_div['dividend_payout'].fillna(0).clip(0, 80), 0, 80)
s_div['dividend_score'] = (s_div['consistency']*0.6 + s_div['payout']*0.4).round(2)

# --- Trend ---
recent = pl[pl['fiscal_year'] >= pl['fiscal_year'].max() - 3].copy()
records = []
for sym, grp in recent.groupby('company_id'):
    grp = grp.sort_values('fiscal_year')
    if len(grp) < 2:
        records.append({'symbol': sym, 'sales_slope': 0, 'profit_slope': 0})
        continue
    x = np.arange(len(grp), dtype=float)
    ss = np.polyfit(x, grp['sales'].fillna(0).values, 1)[0] if grp['sales'].notna().sum() > 1 else 0
    ps = np.polyfit(x, grp['net_profit'].fillna(0).values, 1)[0] if grp['net_profit'].notna().sum() > 1 else 0
    records.append({'symbol': sym, 'sales_slope': ss, 'profit_slope': ps})
df_tr = pd.DataFrame(records)
s_tr = pd.DataFrame({'symbol': df_tr['symbol']})
s_tr['sales_trend']  = percentile_score(df_tr['sales_slope'])
s_tr['profit_trend'] = percentile_score(df_tr['profit_slope'])
s_tr['trend_score']  = (s_tr['sales_trend']*0.5 + s_tr['profit_trend']*0.5).round(2)

print('All dimension scores computed.')

## Step 3 — Combine with spec weights and assign labels

In [ ]:
WEIGHTS = {
    'profitability_score': 0.25,
    'growth_score':        0.20,
    'leverage_score':      0.15,
    'cashflow_score':      0.20,
    'dividend_score':      0.10,
    'trend_score':         0.10,
}

result = s_profit[['symbol','profitability_score']]
for df in [s_growth, s_lev[['symbol','leverage_score']], s_cf[['symbol','cashflow_score']],
           s_div[['symbol','dividend_score']], s_tr[['symbol','trend_score']]]:
    result = result.merge(df, on='symbol', how='outer')

for col in WEIGHTS:
    result[col] = result[col].fillna(50.0)

result['overall_score'] = sum(result[c] * w for c, w in WEIGHTS.items()).round(2)
result['health_label']  = result['overall_score'].apply(assign_label)
result = result.merge(co[['symbol','company_name','sector']], on='symbol', how='left')

print(result[['symbol','overall_score','health_label']].sort_values('overall_score', ascending=False).head(15).to_string(index=False))
print('\nLabel distribution:')
print(result['health_label'].value_counts())

## Step 4 — Visualise score distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(result['overall_score'], bins=20, color='steelblue', edgecolor='white')
for thresh, label, color in [(85,'EXCELLENT','#22c55e'),(70,'GOOD','#84cc16'),(50,'AVERAGE','#f59e0b'),(35,'WEAK','#f97316')]:
    axes[0].axvline(thresh, color=color, linestyle='--', alpha=0.8, linewidth=1.5)
axes[0].set_title('Distribution of overall health scores')
axes[0].set_xlabel('Overall score')
axes[0].set_ylabel('Company count')

label_counts = result['health_label'].value_counts()
colors_map = {'EXCELLENT':'#22c55e','GOOD':'#84cc16','AVERAGE':'#f59e0b','WEAK':'#f97316','POOR':'#ef4444'}
for_order = ['EXCELLENT','GOOD','AVERAGE','WEAK','POOR']
for_order = [l for l in for_order if l in label_counts.index]
axes[1].barh(for_order, [label_counts.get(l, 0) for l in for_order],
             color=[colors_map[l] for l in for_order], alpha=0.85)
axes[1].set_title('Companies per health label')
axes[1].set_xlabel('Count')
plt.tight_layout()
plt.show()

## Step 5 — Cross-validate 5 well-known companies

In [ ]:
VALIDATE = ['TCS', 'HDFCBANK', 'WIPRO', 'ADANIPOWER', 'APOLLOHOSP']
val = result[result['symbol'].isin(VALIDATE)].set_index('symbol')
score_cols = list(WEIGHTS.keys()) + ['overall_score']
print('=== Validation: 5 well-known companies ===')
print(val[['company_name','health_label'] + score_cols].round(1).to_string())

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(VALIDATE))
bottom = np.zeros(len(VALIDATE))
col_colors = ['#3b82f6','#10b981','#8b5cf6','#f59e0b','#ef4444','#ec4899']
for i, (col, w) in enumerate(WEIGHTS.items()):
    vals_w = [val.loc[s, col] * w if s in val.index else 0 for s in VALIDATE]
    ax.bar(x, vals_w, bottom=bottom, label=f"{col.replace('_score','')} ({int(w*100)}%)", color=col_colors[i], alpha=0.85)
    bottom += np.array(vals_w)
ax.set_xticks(x)
ax.set_xticklabels(VALIDATE)
ax.set_ylabel('Weighted score contribution')
ax.set_title('Score breakdown — 5 known companies')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Step 6 — Sensitivity analysis: weight perturbation

In [ ]:
BASE_WEIGHTS = dict(WEIGHTS)
sensitivity = {}
for dim in WEIGHTS:
    test_w = dict(BASE_WEIGHTS)
    test_w[dim] = min(test_w[dim] + 0.05, 1.0)
    total = sum(test_w.values())
    test_w = {k: v/total for k, v in test_w.items()}
    new_scores = sum(result[c] * w for c, w in test_w.items())
    sensitivity[dim] = (new_scores - result['overall_score']).abs().mean()

fig, ax = plt.subplots(figsize=(10, 4))
dims = list(sensitivity.keys())
impacts = [sensitivity[d] for d in dims]
colors_s = ['#3b82f6','#10b981','#8b5cf6','#f59e0b','#ef4444','#ec4899']
ax.bar([d.replace('_score','') for d in dims], impacts, color=colors_s, alpha=0.85)
ax.set_title('Sensitivity: avg score change from +5% weight per dimension')
ax.set_ylabel('Mean absolute score change')
plt.tight_layout()
plt.show()
print('Most sensitive dimension:', max(sensitivity, key=sensitivity.get).replace('_score',''))

## Step 7 — Export scores to CSV

In [ ]:
out_cols = ['symbol','company_name','sector','overall_score','health_label'] + list(WEIGHTS.keys())
result[out_cols].to_csv('../data/clean/health_scores_notebook.csv', index=False)
print('Exported to data/clean/health_scores_notebook.csv')
print(f'Total companies scored: {len(result)}')